### Dataset Used: M5 Forecasting - Accuracy

#### Files
- calendar.csv - Contains information about the dates on which the products are sold.
- sales_train_validation.csv - Contains the historical daily unit sales data per product and store [d_1 - d_1913]
- sample_submission.csv - The correct format for submissions. Reference the Evaluation tab for more info.
- sell_prices.csv - Contains information about the price of the products sold per store and date.
- sales_train_evaluation.csv - Includes sales [d_1 - d_1941] (labels used for the Public leaderboard)


In [0]:
%sql
SHOW TABLES IN workspace.database;

database,tableName,isTemporary
database,calendar,false
database,sales_train_evaluation,false
database,sales_train_validation,false
database,sample_submission,false
database,sell_prices,false
,calendarweeks_2015,true
,calendarweeks_2016,true
,prices_ca1_foods_2015,true


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW Calendar_2015 AS
SELECT *
FROM workspace.database.calendar
WHERE year = 2015;


In [0]:
%sql
SELECT COUNT(*) AS cal_rows FROM Calendar_2015;


cal_rows
365


In [0]:
%sql
SELECT * FROM Calendar_2015 LIMIT 10;

date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
2015-01-01,11448,Thursday,6,1,2015,d_1434,NewYear,National,null,null,1,1,0
2015-01-02,11448,Friday,7,1,2015,d_1435,null,null,null,null,1,0,1
2015-01-03,11449,Saturday,1,1,2015,d_1436,null,null,null,null,1,1,1
2015-01-04,11449,Sunday,2,1,2015,d_1437,null,null,null,null,1,0,0
2015-01-05,11449,Monday,3,1,2015,d_1438,null,null,null,null,1,1,1
2015-01-06,11449,Tuesday,4,1,2015,d_1439,null,null,null,null,1,1,1
2015-01-07,11449,Wednesday,5,1,2015,d_1440,OrthodoxChristmas,Religious,null,null,1,1,0
2015-01-08,11449,Thursday,6,1,2015,d_1441,null,null,null,null,1,0,1
2015-01-09,11449,Friday,7,1,2015,d_1442,null,null,null,null,1,1,1
2015-01-10,11450,Saturday,1,1,2015,d_1443,null,null,null,null,1,0,0


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW CalendarWeeks_2015 AS
SELECT DISTINCT wm_yr_wk
FROM Calendar_2015;

SELECT COUNT(DISTINCT wm_yr_wk) AS weeks_2015 FROM CalendarWeeks_2015;


weeks_2015
53


Building 2016 Sell Prices Table

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW Sell_Prices_2015 AS
SELECT sp.*
FROM workspace.database.sell_prices sp
JOIN CalendarWeeks_2015 w
  ON sp.wm_yr_wk = w.wm_yr_wk;

In [0]:
%sql
--checking duplicated
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT CONCAT(store_id,'|',item_id,'|',wm_yr_wk)) AS distinct_item_week
FROM Sell_Prices_2015;

rows,distinct_item_week
1601188,1601188


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW Prices_CA1_FOODS_2015 AS
SELECT *
FROM Sell_Prices_2015
WHERE store_id = 'CA_1'
  AND item_id LIKE 'FOODS%';

In [0]:
%sql
SELECT COUNT(DISTINCT wm_yr_wk) AS weeks_covered
FROM Prices_CA1_FOODS_2015;


weeks_covered
53


Price Behavior Analysis

In [0]:

%sql
SELECT COUNT(DISTINCT item_id) FROM Prices_CA1_FOODS_2015;

COUNT(DISTINCTitem_id)
1435


In [0]:
%sql
-- No of weeks
SELECT COUNT(DISTINCT wm_yr_wk) FROM Prices_CA1_FOODS_2015;

COUNT(DISTINCTwm_yr_wk)
53


In [0]:
%sql
-- Total Row
SELECT COUNT(*) FROM Prices_CA1_FOODS_2015;

COUNT(*)
75666


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW Prices_CA1_FOODS_2015_Features AS
SELECT
  store_id,
  item_id,
  wm_yr_wk,
  sell_price,
  LAG(sell_price) OVER (
    PARTITION BY store_id, item_id
    ORDER BY wm_yr_wk
  ) AS prev_price,
  sell_price - LAG(sell_price) OVER (
    PARTITION BY store_id, item_id
    ORDER BY wm_yr_wk
  ) AS price_change
FROM Prices_CA1_FOODS_2015;


In [0]:
%sql
-- Checking if the prices even changed?
SELECT COUNT(*) AS change_rows
FROM Prices_CA1_FOODS_2015_Features 
WHERE price_change IS NOT NULL
  AND price_change <> 0;


change_rows
674


In [0]:
%sql
-- How many items prices got changed?
SELECT COUNT(DISTINCT item_id) AS changed_items
FROM Prices_CA1_FOODS_2015_Features 
WHERE price_change <>0;

changed_items
450


In [0]:
%sql 
-- what percent of items price got changed?
SELECT COUNT(DISTINCT CASE WHEN price_change <>0 THEN item_id END) * 100.0/COUNT(DISTINCT item_id) AS pct_items_changed
FROM Prices_CA1_FOODS_2015_Features ;

pct_items_changed
31.35888501742160


In [0]:
%sql 
-- Top 10 items by number of price changes
-- checking the volatility is spread out or concentrated
SELECT item_id, COUNT(*) AS change_count
FROM Prices_CA1_FOODS_2015_Features 
WHERE price_change <>0
GROUP BY item_id
ORDER BY change_count DESC
LIMIT 10;

item_id,change_count
FOODS_3_589,7
FOODS_3_707,7
FOODS_3_205,7
FOODS_3_647,7
FOODS_3_326,4
FOODS_1_034,4
FOODS_3_077,4
FOODS_1_095,4
FOODS_1_084,3
FOODS_1_096,3


In [0]:
%sql 
--checking how big were the changes?
SELECT AVG(ABS(price_change)) AS avg_change
FROM Prices_CA1_FOODS_2015_Features 
WHERE price_change <>0;

avg_change
0.34958456973293767


In [0]:
%sql
SELECT 'Calendar_2015' AS name, COUNT(*) AS rows FROM Calendar_2015
UNION ALL
SELECT 'Sell_Prices_2015', COUNT(*) FROM Sell_Prices_2015
UNION ALL
SELECT 'Prices_CA1_FOODS_2015 ', COUNT(*) FROM Prices_CA1_FOODS_2015 
UNION ALL
SELECT 'Prices_CA1_FOODS_2015_Features ', COUNT(*) FROM Prices_CA1_FOODS_2015_Features ;


name,rows
Calendar_2015,365
Sell_Prices_2015,1601188
Prices_CA1_FOODS_2015,75666
Prices_CA1_FOODS_2015_Features,75666
